In [54]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

In [55]:
# inbuilt tools
import wikipedia
wikipedia.set_user_agent("GenerativeAINotesBot/1.0 (contact@example.com)")

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

print(wiki.name)

wikipedia


In [56]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=300)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

print(arxiv.name)

arxiv


In [57]:
# custom tools -> retriever_tool [RAG TOOL]
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [58]:
loader = WebBaseLoader("https://www.langchain.com/blog/introducing-langserve")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5509.39it/s]


In [59]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="Langserve-search",
    description="Search any information about Langserve."
)

print(retriever_tool.name)

Langserve-search


In [60]:
tools = [wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Desktop\\AI\\Generative-ai-notes\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=300)),
 StructuredTool(name='Langserve-search', description='Search any information about Langserve.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000024C3BACCD60>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000024C3BACE840>)]

In [61]:
# Run all this tools with Agents and LLM Models

from langchain_google_genai import ChatGoogleGenerativeAI

import os
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [62]:
# PROMPT
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [63]:
prompt.get_prompts()

[PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')]

In [64]:
from langchain_classic.agents import create_react_agent
agent = create_react_agent(
    model,
    tools,
    prompt=prompt
)

In [65]:
# Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True, handle_parsing_errors=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': "wikipedia - A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\narxiv - A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.\nLangserve-search(query: 'str', callbacks: 'Callbacks' = None) -> 'str | tuple[str, list[Document]]' - Search any information about Langserve.", 'tool_names': 'wikipedia, arxiv, Langserve-search'}, metadata={'lc_hub_owner':

In [66]:
agent_executor.invoke({"input":"What is machine learning?"})



> Entering new AgentExecutor chain...
Action: wikipedia
Action Input: Machine learningPage: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in theThought: I now know the final answer based on the Wikipedia search.
Final Answer: Machine learning (ML) is a field of study in artificial intelligence (AI) focused on developing and studying statistical algorithms that can learn from data and generalize to unseen data. This allows computers to perform specific tasks and make decisions or predictions without being explicitly programmed to do so.

> Finished chain.


{'input': 'What is machine learning?',
 'output': 'Machine learning (ML) is a field of study in artificial intelligence (AI) focused on developing and studying statistical algorithms that can learn from data and generalize to unseen data. This allows computers to perform specific tasks and make decisions or predictions without being explicitly programmed to do so.'}

In [ ]:
agent_executor.invoke({"input":"Why Langserve exists?"})

In [ ]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})